In [1]:
import numpy as np
import keras
from keras import layers
import os
import cv2
import matplotlib.pyplot as plt


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wanderdust/coin-images")

print("Path to dataset files:", path)

C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|███████████████████████████████████████████████████████████████████████████████| 459M/459M [00:23<00:00, 20.9MB/s]

Extracting files...


Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\wanderdust\coin-images\versions\1


In [3]:
inputs = keras.Input(shape=(784,))

In [7]:
dense = layers.Dense(64, activation="relu")
x = dense(inputs)

In [8]:
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(10)(x)

In [20]:
dataset_path = r"C:\Users\USER\.cache\kagglehub\datasets\wanderdust\coin-images\versions\1"

for root, dirs, files in os.walk(dataset_path):
    depth = root.replace(dataset_path, "").count(os.sep)
    if depth <= 3:
        print(f"{'  ' * depth}{os.path.basename(root)}/  ({len(files)} files)")

1/  (1 files)
  coins/  (0 files)
    data/  (1 files)
      test/  (0 files)
      train/  (0 files)
      validation/  (0 files)


In [21]:
import os, json, glob

json_path = glob.glob(os.path.join(dataset_path, "**", "cat_to_name.json"), recursive=True)
print(json_path)

with open(json_path[0]) as f:
    cat_to_name = json.load(f)

['C:\\Users\\USER\\.cache\\kagglehub\\datasets\\wanderdust\\coin-images\\versions\\1\\cat_to_name.json', 'C:\\Users\\USER\\.cache\\kagglehub\\datasets\\wanderdust\\coin-images\\versions\\1\\coins\\data\\cat_to_name.json']


In [22]:
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")
test_dir  = os.path.join(data_dir, "test")

classes = sorted(os.listdir(train_dir))
print(len(classes), classes[:5])
print(len(os.listdir(os.path.join(train_dir, classes[0]))), "images in first class")

211 ['1', '10', '100', '101', '102']
28 images in first class


In [23]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Adjust if your structure differs (e.g. there may be an extra "data" folder)
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")   # may be "valid"
test_dir  = os.path.join(data_dir, "test")

IMG_SIZE = 224
BATCH_SIZE = 32 #how mant images go through the model at finetuning. How many are we looking at before updating the weights,
#batch size is to look at when were looking at training time and resources

# ImageNet stats - use these if you plan to use a pretrained model
#Noramlise computes (pixel - mean) / std
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

#rotate, flip, and color jitter make the image look a bit different every time to make the model learn what a coin looks like, rrather than what the images look like
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),          # coins can be any orientation
    transforms.RandomHorizontalFlip(), # play with this. coins dont flip
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), #converts pixels from 0-255 to 0-1 and rearranged image into a tensor
    transforms.Normalize(mean, std), #shifts and scales the tensor
])

#we don't need random augmentation for the evaluation and testing sets
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
#the transforms are stored in the ImageFolder and applied each time the image is loaded. This ensures the random augmentations are different every epoch
train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

#wraps a dataset and hands it off to the model, armed with all the level settings we decided on
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Classes:", len(train_ds.classes)) #counts number of classes
print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))

Classes: 211
Train/Val/Test sizes: 6413 844 844


In [25]:
from collections import Counter
counts = Counter(train_ds.targets)
print(min(counts.values()), max(counts.values()))

8 52


In [26]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [28]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(36)

In [30]:
import torch.nn as nn
from torchvision import models

num_classes = len(train_ds.classes)

#resnet is pretrained on imagenet, so it knows to recognize features of an image
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for p in model.parameters():
    #requires_grad flags each parameter on whether or not to compute gradients for it.
    #were setting it to false for now so that they're excluded from learning to keep pretrained knowledge in tact and save cpu
    #consider playing with it
    p.requires_grad = False          # freeze the pretrained backbone

#model fc (fully connected) is the last layer of resnet, turns the features into class scores
#this is the only part that learns
model.fc = nn.Linear(model.fc.in_features, num_classes)   # new head for 211 coins
model = model.to(device)

#how wrong is the model? 
criterion = nn.CrossEntropyLoss()
#tries to minimize the wrongness
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [32]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader: #x - images, a tensor shaped [32, 3, 224, 224], y - true labels, tensor or 32 class indices
            x, y = x.to(device), y.to(device) #both sent to device
            out = model(x)
            loss = criterion(out, y) #average loss per image in batch
            if train:
                optimizer.zero_grad() #clear the old gradients so they dont confuse the new calculations
                loss.backward() #compute new gradients (backpropagation)
                optimizer.step() #updates weights
            total_loss += loss.item() * x.size(0) #average losses * images in the batch = total losses
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

for epoch in range(5):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f}")

Epoch 1: train acc 0.272 | val loss 3.571 | val acc 0.448
Epoch 2: train acc 0.634 | val loss 2.794 | val acc 0.575
Epoch 3: train acc 0.738 | val loss 2.350 | val acc 0.614
Epoch 4: train acc 0.792 | val loss 2.047 | val acc 0.654
Epoch 5: train acc 0.831 | val loss 1.860 | val acc 0.669


In [42]:
results = {}
results["with_flip"] = {"val_acc": va, "val_loss": vl}
print(results)

{'with_flip': {'val_acc': 0.669431279620853, 'val_loss': 1.8604557898372271}}


This time we'll run everything exactly the same, except with no horizontal flips

In [35]:
# Adjust if your structure differs (e.g. there may be an extra "data" folder)
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")   # may be "valid"
test_dir  = os.path.join(data_dir, "test")

IMG_SIZE = 224
BATCH_SIZE = 32 #how mant images go through the model at finetuning. How many are we looking at before updating the weights,
#batch size is to look at when were looking at training time and resources

# ImageNet stats - use these if you plan to use a pretrained model
#Noramlise computes (pixel - mean) / std
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

#rotate, flip, and color jitter make the image look a bit different every time to make the model learn what a coin looks like, rrather than what the images look like
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),          # coins can be any orientation
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), #converts pixels from 0-255 to 0-1 and rearranged image into a tensor
    transforms.Normalize(mean, std), #shifts and scales the tensor
])

#we don't need random augmentation for the evaluation and testing sets
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
#the transforms are stored in the ImageFolder and applied each time the image is loaded. This ensures the random augmentations are different every epoch
train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

#wraps a dataset and hands it off to the model, armed with all the level settings we decided on
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Classes:", len(train_ds.classes)) #counts number of classes
print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))

Classes: 211
Train/Val/Test sizes: 6413 844 844


In [36]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(36)

In [37]:
num_classes = len(train_ds.classes)

#resnet is pretrained on imagenet, so it knows to recognize features of an image
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for p in model.parameters():
    #requires_grad flags each parameter on whether or not to compute gradients for it.
    #were setting it to false for now so that they're excluded from learning to keep pretrained knowledge in tact and save cpu
    #consider playing with it
    p.requires_grad = False          # freeze the pretrained backbone

#model fc (fully connected) is the last layer of resnet, turns the features into class scores
#this is the only part that learns
model.fc = nn.Linear(model.fc.in_features, num_classes)   # new head for 211 coins
model = model.to(device)

#how wrong is the model? 
criterion = nn.CrossEntropyLoss()
#tries to minimize the wrongness
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [38]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader: #x - images, a tensor shaped [32, 3, 224, 224], y - true labels, tensor or 32 class indices
            x, y = x.to(device), y.to(device) #both sent to device
            out = model(x)
            loss = criterion(out, y) #average loss per image in batch
            if train:
                optimizer.zero_grad() #clear the old gradients so they dont confuse the new calculations
                loss.backward() #compute new gradients (backpropagation)
                optimizer.step() #updates weights
            total_loss += loss.item() * x.size(0) #average losses * images in the batch = total losses
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

for epoch in range(5):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f}")

Epoch 1: train acc 0.274 | val loss 3.549 | val acc 0.457
Epoch 2: train acc 0.645 | val loss 2.696 | val acc 0.556
Epoch 3: train acc 0.754 | val loss 2.320 | val acc 0.608
Epoch 4: train acc 0.805 | val loss 2.015 | val acc 0.661
Epoch 5: train acc 0.840 | val loss 1.840 | val acc 0.659


In [43]:
results["_flip"] = {"val_acc": va, "val_loss": vl}
print(results)

{'with_flip': {'val_acc': 0.669431279620853, 'val_loss': 1.8604557898372271}, 'no_flip': {'val_acc': 0.6587677725118484, 'val_loss': 1.8402982469983575}}


In [45]:
for p in model.parameters():
    p.requires_grad = True

backbone = [p for n, p in model.named_parameters() if not n.startswith("fc")]
optimizer = torch.optim.Adam([
    {"params": backbone,            "lr": 3e-5},
    {"params": model.fc.parameters(), "lr": 3e-4},
])

import time
for epoch in range(5):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")
    if epoch == 0 or va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_coin_model.pt")

Epoch 1: train acc 0.977 | val loss 1.128 | val acc 0.780 | 1674s
Epoch 2: train acc 0.981 | val loss 1.107 | val acc 0.797 | 3101s
Epoch 3: train acc 0.981 | val loss 1.092 | val acc 0.788 | 1901s
Epoch 4: train acc 0.984 | val loss 1.104 | val acc 0.802 | 2116s
Epoch 5: train acc 0.982 | val loss 1.118 | val acc 0.799 | 1931s


In [46]:
torch.save(model.state_dict(), "best_coin_model.pt")

In [47]:
model.eval()
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"Test loss {test_loss:.3f} | Test acc {test_acc:.3f}")

Test loss 0.873 | Test acc 0.842


In [48]:
with open(os.path.join(data_dir, "cat_to_name.json")) as f:
    cat_to_name = json.load(f)

class_names = [cat_to_name[c] for c in train_ds.classes]
print(len(class_names), class_names[:3])

211 ['1 Cent,Australian dollar,australia', '5 Centavos,Brazilian Real,brazil', '5 Won,Korean Won,south_korea']


In [49]:
from collections import defaultdict, Counter

counts = Counter(train_ds.targets)      # training images per class
correct = defaultdict(int)
total = defaultdict(int)

model.eval()
with torch.no_grad():
    for x, y in val_loader:
        preds = model(x.to(device)).argmax(1).cpu()
        for t, p in zip(y, preds):
            total[t.item()] += 1
            correct[t.item()] += int(t == p)

# 10 worst classes
worst = sorted(total, key=lambda c: correct[c] / total[c])[:10]
for c in worst:
    print(f"{class_names[c]:<40} {correct[c]}/{total[c]}   train imgs: {counts[c]}")

2 euro Cent,Euro,spain                   0/4   train imgs: 28
5 Kopeks,Russian Ruble,russia            1/4   train imgs: 28
5 Dollars,Singapore Dollar,singapore     1/4   train imgs: 8
1 Franc,Swiss Franc,switzerland          1/4   train imgs: 14
2 Francs,Swiss Franc,switzerland         1/4   train imgs: 28
1 Satang,Thai Baht,thailand              1/4   train imgs: 24
5 Satang,Thai Baht,thailand              1/4   train imgs: 24
1 Krone,Danish Krone,denmark             1/4   train imgs: 28
5 Kroner,Danish Krone,denmark            1/4   train imgs: 28
1 Euro,Euro,spain                        1/4   train imgs: 28


In [50]:
low  = [c for c in total if counts[c] <= 15]
rest = [c for c in total if counts[c] > 15]

def acc(classes):
    return sum(correct[c] for c in classes) / sum(total[c] for c in classes)

print(f"Coins with <=15 train images: {len(low)} classes, acc {acc(low):.3f}")
print(f"Coins with  >15 train images: {len(rest)} classes, acc {acc(rest):.3f}")

Coins with <=15 train images: 4 classes, acc 0.625
Coins with  >15 train images: 207 classes, acc 0.802


In [59]:
from PIL import Image
import torch.nn.functional as F

def predict(image_path, top_k=3):
    img = Image.open(image_path).convert("RGB")
    x = eval_transform(img).unsqueeze(0).to(device)    # add batch dimension

    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0]

    top_p, top_i = probs.topk(top_k)
    for p, i in zip(top_p, top_i):
        print(f"{class_names[i]:<40} {p.item()*100:.1f}%")

print("5 shek")
predict(r"C:\Users\USER\Downloads\20261019_195006.jpg")
print("1 shek")
predict(r"C:\Users\USER\Downloads\20261019_194945.jpg")

5 shek
5 Won,Korean Won,south_korea             34.9%
1 Kopek,Russian Ruble,russia             21.8%
50 Kopeks,Russian Ruble,russia           10.8%
1 shek
1 2 New Sheqel,Israeli New Shekel,israel 54.4%
1 New Sheqel,Israeli New Shekel,israel   14.3%
1 Yuan,Chinese Yuan Renminbi,china       9.3%


In [57]:
all_true, all_pred = [], []
model.eval()
with torch.no_grad():
    for x, y in val_loader:
        all_pred += model(x.to(device)).argmax(1).cpu().tolist()
        all_true += y.tolist()

for c in worst:
    wrong = [class_names[p] for t, p in zip(all_true, all_pred) if t == c and p != c]
    print(class_names[c])
    for w in wrong:
        print("   predicted:", w)

2 euro Cent,Euro,spain
   predicted: 2 Euro,Euro,spain
   predicted: 2 Euro,Euro,spain
   predicted: 5 euro Cent,Euro,spain
   predicted: 20 Forint,Hungarian Forint,hungary
5 Kopeks,Russian Ruble,russia
   predicted: 5 Cents,South African Rand,south_africa
   predicted: 1 Kurus,Turkish Lira,turkey
   predicted: 1 Cent,US Dollar,usa
5 Dollars,Singapore Dollar,singapore
   predicted: 10 Kroner,Danish Krone,denmark
   predicted: 10 Dollars,taiwan Dollar,taiwan
   predicted: 5 Jiao,Chinese Yuan Renminbi,china
1 Franc,Swiss Franc,switzerland
   predicted: 2 Francs,Swiss Franc,switzerland
   predicted: 2 Francs,Swiss Franc,switzerland
   predicted: 1 2 Franc,Swiss Franc,switzerland
2 Francs,Swiss Franc,switzerland
   predicted: 1 2 Franc,Swiss Franc,switzerland
   predicted: 10 Rappen,Swiss Franc,switzerland
   predicted: 1 2 Franc,Swiss Franc,switzerland
1 Satang,Thai Baht,thailand
   predicted: 25 Satang,Thai Baht,thailand
   predicted: 1 Baht,Thai Baht,thailand
   predicted: 10 Dollars,ta